# Data inspection cleaning and score construction

**Project:** Friendship contact and loneliness in the EU Loneliness Survey 2022.

This notebook prepares the data for Week 6 exploratory data analysis (EDA). It adapts the inspection commands from Week 3 and the documented cleaning approach from Week 4. Run the cells in order and read their outputs.

**Setup:** Place this notebook in a `notebooks` folder and both original CSV files in `data` within the same project folder. Select a Python environment with pandas and Jupyter installed in VS Code. Open the project folder, then run this notebook. The path cell supports running from either the project root or its notebooks folder.

**Decisions used here:** treat documented non-response as missing; calculate a three-item loneliness sum only when all three answers are valid; use a conservative main sample with confirmed age 16+ and all five core responses. Retain other cases in the cleaned file with flags so these decisions can be revisited. Do not impute responses or apply survey weights here. The age policy is our project decision, not a claim about the survey publisher's recommended exclusion rules.

Sources: [JRC dataset](https://doi.org/10.2905/JRC.V4VT8T8) and [EU27 questionnaire](https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/EU-Loneliness-Survey/eu_loneliness_survey_quest_eu27.pdf), questions 38 and 44. The values and labels CSVs are two representations of the same respondents; do not stack them.

## Expanded version
The main question remains friendship contact and loneliness. Additional variables describe social networks, available support, family contact, participation and personal circumstances. They support later subgroup exploration; they are not automatically model controls.

Optional missing values do not exclude respondents from the main analysis. Each contextual comparison must report its own valid sample size. This notebook prepares variables and audit tables; the expanded comparisons belong in Notebook 2.


## 1 Import libraries and locate files

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
from IPython.display import display

# Change PROJECT_ROOT if your project is stored elsewhere.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data"
VALUES_NAME = "eu_loneliness_survey_eu27_values.csv"
LABELS_NAME = "eu_loneliness_survey_eu27_labels.csv"

VALUES_PATH = RAW_DIR / VALUES_NAME
LABELS_PATH = RAW_DIR / LABELS_NAME
for path in [VALUES_PATH, LABELS_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f"Cannot find {path}. Check RAW_DIR in this cell.")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "results" / "tables"
for folder in [PROCESSED_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Fingerprints let us verify that the raw files remain unchanged.
raw_hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
              for p in [VALUES_PATH, LABELS_PATH]}
print("Raw data folder:", RAW_DIR)

Raw data folder: c:\Users\ladde\Documents\GitHub\PresentationDraft1\data


## 2 Load and inspect the dataset
`low_memory=False` avoids mixed-type inference across file chunks. It does not clean the data. We display only project-relevant fields rather than unrelated sensitive responses.

In [2]:
raw = pd.read_csv(VALUES_PATH, low_memory=False)
labels = pd.read_csv(LABELS_PATH, low_memory=False)

contact_cols = ["friends_meet_face", "friends_meet_tele"]
ucla_cols = ["loneliness_ucla_a", "loneliness_ucla_b", "loneliness_ucla_c"]
core_cols = contact_cols + ucla_cols
weight_cols = ["w_country_5", "w_country", "w_population", "w_eu27_5", "w_eu27"]

context_cols = [
    "gender", "relationship", "work_status", "education", "municipality_type",
    "health_general", "health_condition",
    "social_support_a", "social_support_b", "social_support_c", "social_support_d",
    "family_meet_face", "family_meet_tele", "social_activities_a",
    "friends_n__1", "friends_n__1__open", "family_n__1", "family_n__1__open",
    "adults_n", "childr_n_0to5", "childr_n_6to15", "others_n",
]

selected_cols = ["id_pers", "country", "age"] + core_cols + context_cols + weight_cols

assert set(selected_cols).issubset(raw.columns), "A required column is missing."
assert raw.columns.equals(labels.columns), "The file columns differ."
assert raw["id_pers"].equals(labels["id_pers"]), "Respondents are not aligned."
assert raw["id_pers"].notna().all(), "Investigate missing respondent IDs."
assert raw["id_pers"].is_unique, "Investigate repeated IDs before proceeding."

print("Rows and columns:", raw.shape)
print("Countries:", raw["country"].nunique())
print("Exact duplicate rows:", raw.duplicated().sum())
display(raw[selected_cols].head())
raw[selected_cols].info()
display(raw[core_cols + ["age"]].describe())

Rows and columns: (25646, 208)
Countries: 27
Exact duplicate rows: 0


,id_pers,country,age,friends_meet_face,friends_meet_tele,loneliness_ucla_a,loneliness_ucla_b,loneliness_ucla_c,gender,relationship,...,family_n__1__open,adults_n,childr_n_0to5,childr_n_6to15,others_n,w_country_5,w_country,w_population,w_eu27_5,w_eu27
0,1,1,32,6,6,1,2,2,1,1,...,2.0,3.0,0,0.0,0,0.423861,0.359538,0.512764,0.217341,0.184358
1,2,1,56,3,4,2,1,1,1,1,...,1.0,3.0,0,0.0,2,0.354297,0.352568,0.512764,0.181671,0.180784
2,3,1,45,3,3,1,1,1,1,1,...,2.0,1.0,0,0.0,0,1.558302,2.228209,0.512764,0.799041,1.142545
3,4,1,35,5,7,1,2,1,2,2,...,4.0,4.0,0,0.0,0,0.614236,0.556907,0.512764,0.314958,0.285561
4,5,1,43,5,6,3,1,1,1,3,...,9.0,2.0,0,1.0,0,4.200130,2.228209,0.512764,2.153674,1.142545


<class 'pandas.DataFrame'>
RangeIndex: 25646 entries, 0 to 25645
Data columns (total 35 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id_pers              25646 non-null  int64  
 1   country              25646 non-null  int64  
 2   age                  25646 non-null  int64  
 3   friends_meet_face    25646 non-null  int64  
 4   friends_meet_tele    25646 non-null  int64  
 5   loneliness_ucla_a    25646 non-null  int64  
 6   loneliness_ucla_b    25646 non-null  int64  
 7   loneliness_ucla_c    25646 non-null  int64  
 8   gender               25646 non-null  int64  
 9   relationship         25646 non-null  int64  
 10  work_status          25646 non-null  int64  
 11  education            25646 non-null  int64  
 12  municipality_type    25646 non-null  int64  
 13  health_general       25646 non-null  int64  
 14  health_condition     25646 non-null  int64  
 15  social_support_a     25646 non-null  int64  
 1

,friends_meet_face,friends_meet_tele,loneliness_ucla_a,loneliness_ucla_b,loneliness_ucla_c,age
count,25646.000000,25646.000000,25646.000000,25646.000000,25646.000000,25646.000000
mean,22.802035,21.248265,17.796187,21.143492,18.374483,44.768229
std,134.975325,126.023870,125.535131,138.173247,128.060314,25.599794
min,1.000000,1.000000,1.000000,1.000000,1.000000,15.000000
25%,3.000000,4.000000,1.000000,1.000000,1.000000,32.000000
50%,4.000000,5.000000,2.000000,1.000000,2.000000,44.000000
75%,6.000000,7.000000,2.000000,2.000000,2.000000,55.000000
max,999.000000,999.000000,999.000000,999.000000,999.000000,997.000000


## 3 Match codes to labels
Read the labels before recoding. Blank checks alone miss special numeric codes such as 999. The following mappings come directly from matching rows in the two supplied files.

In [3]:
mapping_tables = []
for col in core_cols + ["age"] + context_cols:
    mapping = pd.DataFrame({"code": raw[col], "label": labels[col]})
    mapping = mapping.value_counts(dropna=False).reset_index(name="n")
    mapping.insert(0, "variable", col)
    mapping_tables.append(mapping)
    if col in core_cols:
        print(col)
        display(mapping.sort_values("code"))

codebook_observed = pd.concat(mapping_tables, ignore_index=True)
print("Unusual age codes:")
display(codebook_observed.loc[
    (codebook_observed["variable"] == "age") &
    ((codebook_observed["code"] < 16) | (codebook_observed["code"] == 997))
])

friends_meet_face


,variable,code,label,n
6,friends_meet_face,1,Never,1184
2,friends_meet_face,2,Every two months or less frequently,4189
1,friends_meet_face,3,Once a month,4487
3,friends_meet_face,4,Every two weeks,3725
0,friends_meet_face,5,Every week,5517
4,friends_meet_face,6,More than once a week,3687
5,friends_meet_face,7,Daily,2376
7,friends_meet_face,999,Prefer not to say,481


friends_meet_tele


,variable,code,label,n
6,friends_meet_tele,1,Never,1001
5,friends_meet_tele,2,Every two months or less frequently,1986
4,friends_meet_tele,3,Once a month,2580
3,friends_meet_tele,4,Every two weeks,2747
1,friends_meet_tele,5,Every week,5263
2,friends_meet_tele,6,More than once a week,5214
0,friends_meet_tele,7,Daily,6436
7,friends_meet_tele,999,Prefer not to say,419


loneliness_ucla_a


...

,variable,code,label,n
0,loneliness_ucla_b,1,Hardly ever or never,12996
1,loneliness_ucla_b,2,Some of the time,8692
2,loneliness_ucla_b,3,Often,3456
3,loneliness_ucla_b,999,Prefer not to say,502


loneliness_ucla_c


,variable,code,label,n
0,loneliness_ucla_c,1,Hardly ever or never,12471
1,loneliness_ucla_c,2,Some of the time,9044
2,loneliness_ucla_c,3,Often,3701
3,loneliness_ucla_c,999,Prefer not to say,430


Unusual age codes:


,variable,code,label,n
96,age,997.0,Implausible value,12
102,age,15.0,15,3


## 4 Select variables and clean documented non-response
The original dataframe and CSV files remain unchanged. In these five core columns, 999 means prefer not to say. Check unexpected codes explicitly instead of silently treating every unfamiliar number as missing. Do not replace 999 across the entire dataset: codes have variable-specific meanings.

In [4]:
clean = raw[selected_cols].copy()
audit_rows = []

for col in core_cols:
    allowed = list(range(1, 8)) if col in contact_cols else [1, 2, 3]
    unexpected = clean[col].notna() & ~clean[col].isin(allowed + [999])
    if unexpected.any():
        raise ValueError(f"Unexpected codes in {col}: {clean.loc[unexpected, col].unique()}")
    audit_rows.append({
        "variable": col,
        "blank_before": int(clean[col].isna().sum()),
        "prefer_not_to_say": int(clean[col].eq(999).sum()),
    })
    clean[col] = clean[col].replace(999, float("nan"))

missing_audit = pd.DataFrame(audit_rows).set_index("variable")
missing_audit["missing_after"] = clean[core_cols].isna().sum()
missing_audit["missing_percent"] = (100 * clean[core_cols].isna().mean()).round(2)
display(missing_audit)

,blank_before,prefer_not_to_say,missing_after,missing_percent
variable,,,,
friends_meet_face,0,481,481,1.88
friends_meet_tele,0,419,419,1.63
loneliness_ucla_a,0,413,413,1.61
loneliness_ucla_b,0,502,502,1.96
loneliness_ucla_c,0,430,430,1.68


## 5 Handle age transparently
The labels file identifies 997 as an implausible value. Recode it as missing and preserve the original age code. Keep age 15 as recorded; do not guess a replacement. For our main 16+ analysis, require a known age of at least 16. This conservatively excludes missing/implausible age and recorded under-16 cases. We retain all rows in the cleaned file so alternative decisions can be examined later.

In [5]:
clean["age_raw"] = clean["age"]
clean["age"] = clean["age"].replace(997, float("nan"))
clean["age_missing"] = clean["age"].isna()
clean["age_under_16"] = clean["age"].lt(16)
clean["age_eligible"] = clean["age"].notna() & clean["age"].ge(16)

age_audit = pd.DataFrame({
    "condition": ["Missing or implausible age", "Recorded age below 16", "Confirmed age 16+"],
    "n": [clean["age_missing"].sum(), clean["age_under_16"].sum(), clean["age_eligible"].sum()]
})
display(age_audit)

,condition,n
0,Missing or implausible age,12
1,Recorded age below 16,3
2,Confirmed age 16+,25631


## 6 Prepare contextual categories

We preserve original numeric codes for nominal variables such as gender, relationship status, work status and country, and add readable labels. Their codes must not be treated as numerical ranks in correlations. Education, contact frequency, support and general health are ordered categories; intervals are not assumed equal.

- `social_support_a`: practical help if confined to bed.
- `social_support_b`: someone with whom to share worries and fears.
- `social_support_c`: someone available for enjoyable activities.
- `social_support_d`: someone who makes the respondent feel loved and wanted.
- `social_activities_a`: participation in club, society or association activities.
- `health_general`: higher codes indicate poorer health; support codes instead increase with more available support.

Family-contact questions exclude household members. Close-friend and close-family counts include co-residents. Do not interpret these as exactly matching networks.

Only documented special codes are recoded. We retain the support items separately rather than inventing a combined scale. Long-term health condition status is included; conditional follow-up severity is omitted because its structural missingness would require separate handling.

In [6]:
# Each dictionary entry specifies valid codes and documented missing codes.
category_rules = {
    "gender": ([1, 2, 3], [999]),
    "relationship": (list(range(1, 6)), [999]),
    "work_status": (list(range(1, 9)), [999]),
    "education": (list(range(1, 6)), [999]),
    "municipality_type": ([1, 2, 3], [998]),
    "health_general": (list(range(1, 6)), [998, 999]),
    "health_condition": ([1, 2], [998, 999]),
    "family_meet_face": (list(range(1, 8)), [999]),
    "family_meet_tele": (list(range(1, 8)), [999]),
    "social_activities_a": (list(range(1, 8)), [999]),
}
for col in ["social_support_a", "social_support_b", "social_support_c", "social_support_d"]:
    category_rules[col] = (list(range(1, 6)), [998, 999])

context_audit_rows = []
for col, (valid_codes, missing_codes) in category_rules.items():
    unexpected = clean[col].notna() & ~clean[col].isin(valid_codes + missing_codes)
    if unexpected.any():
        raise ValueError(f"Review unexpected codes in {col}: {clean.loc[unexpected, col].unique()}")
    context_audit_rows.append({
        "variable": col,
        "blank_before": int(clean[col].isna().sum()),
        "dont_know": int(clean[col].eq(998).sum()),
        "prefer_not_to_say": int(clean[col].eq(999).sum()),
    })
    clean[col] = clean[col].replace(missing_codes, float("nan"))
    mapping = pd.DataFrame({"code": raw[col], "label": labels[col]}).drop_duplicates()
    assert mapping["code"].is_unique, f"Ambiguous labels in {col}"
    clean[col + "_label"] = clean[col].map(mapping.set_index("code")["label"])

context_missing_audit = pd.DataFrame(context_audit_rows).set_index("variable")
context_missing_audit["missing_after"] = clean[list(category_rules)].isna().sum()
context_missing_audit["missing_percent"] = (100 * clean[list(category_rules)].isna().mean()).round(2)
display(context_missing_audit)

,blank_before,dont_know,prefer_not_to_say,missing_after,missing_percent
variable,,,,,
gender,0,0,80,80,0.31
relationship,0,0,182,182,0.71
work_status,0,0,0,0,0.00
education,0,0,162,162,0.63
municipality_type,0,263,0,263,1.03
health_general,0,0,338,338,1.32
health_condition,0,2864,654,3518,13.72
family_meet_face,0,0,421,421,1.64
family_meet_tele,0,0,317,317,1.24


## 7 Prepare close relationship counts and household information

The `__1` fields indicate whether a numeric answer was supplied; the `__open` fields contain the count. A valid zero means no close friends/family members, not missing data. Refusals must never become zero. Investigate contradictions between the response flag and count instead of silently fixing them.

Household counts describe budget-sharing adults, children under 16 and other co-residents. We retain the separate counts and do not equate one adult in the household with living alone. Questionnaire-permitted high counts remain unless evidence establishes an error. Non-integer counts (for example 0.8 people) or out-of-range values are flagged and treated as missing for that variable, without rounding or removing the respondent. Source count fields or explicit `_raw` columns preserve the original values.

In [7]:
count_audit_rows = []
for prefix, new_col in [("friends", "close_friends_n"), ("family", "close_family_n")]:
    flag = clean[prefix + "_n__1"]
    count = clean[prefix + "_n__1__open"]
    assert flag.isin([1, 999]).all(), f"Unexpected response flag: {prefix}"
    assert count.loc[flag.eq(999)].isna().all(), f"Count present despite refusal: {prefix}"
    numeric_answer = flag.eq(1)
    valid_count = count.between(0, 100) & count.mod(1).eq(0)
    clean[new_col + "_invalid"] = numeric_answer & ~valid_count
    clean[new_col] = count.where(numeric_answer & valid_count)
    count_audit_rows.append({"variable": new_col, "valid_n": int(clean[new_col].notna().sum()),
                             "missing_n": int(clean[new_col].isna().sum()),
                             "invalid_n": int(clean[new_col + "_invalid"].sum()),
                             "zero_n": int(clean[new_col].eq(0).sum())})

household_ranges = {"adults_n": (1, 30), "childr_n_0to5": (0, 10),
                    "childr_n_6to15": (0, 20), "others_n": (0, 50)}
for col, (low, high) in household_ranges.items():
    present = clean[col].notna()
    valid = clean[col].between(low, high) & clean[col].mod(1).eq(0)
    clean[col + "_raw"] = clean[col]
    clean[col + "_invalid"] = present & ~valid
    clean[col] = clean[col].where(valid)
    count_audit_rows.append({"variable": col, "valid_n": int(clean[col].notna().sum()),
                             "missing_n": int(clean[col].isna().sum()),
                             "invalid_n": int(clean[col + "_invalid"].sum()),
                             "zero_n": int(clean[col].eq(0).sum())})
count_audit = pd.DataFrame(count_audit_rows)
display(count_audit)

# Broad descriptive groups selected in advance, not to maximise differences.
# These are project choices; retain continuous age and original counts too.
clean["age_group"] = pd.cut(clean["age"], bins=[16, 30, 45, 60, float("inf")],
                            labels=["16-29", "30-44", "45-59", "60+"], right=False)
clean["close_friends_group"] = pd.cut(clean["close_friends_n"],
    bins=[-1, 0, 2, 5, 100], labels=["0", "1-2", "3-5", "6+"])

,variable,valid_n,missing_n,invalid_n,zero_n
0,close_friends_n,23649,1997,1,1733
1,close_family_n,23851,1795,2,743
2,adults_n,25645,1,1,0
3,childr_n_0to5,25646,0,0,21311
4,childr_n_6to15,25645,1,1,18215
5,others_n,25646,0,0,22031


## 8 Construct the three-item loneliness sum
Each UCLA item is coded 1 to 3, with higher responses indicating greater loneliness. We define `ucla_total` as their sum, ranging from 3 to 9. All items point in the same direction. We require all three valid answers and make no partial-score substitutions. `min_count=3` is essential: pandas otherwise skips missing values when summing. This is a score, not a diagnosis; we do not invent a clinical cutoff.

In [8]:
clean["ucla_items_answered"] = clean[ucla_cols].notna().sum(axis=1)
clean["ucla_total"] = clean[ucla_cols].sum(axis=1, min_count=3)
clean["core_complete"] = clean[core_cols].notna().all(axis=1)
clean["include_main_analysis"] = clean["age_eligible"] & clean["core_complete"]

analysis = clean.loc[clean["include_main_analysis"]].copy()
print("Rows with all five core responses:", int(clean["core_complete"].sum()))
print("Main analysis sample after age eligibility:", len(analysis))

Rows with all five core responses: 24442
Main analysis sample after age eligibility: 24431


## 9 Record sequential exclusions
Counts in the flow table are sequential, so no person is subtracted twice. Missingness by variable overlaps and must not be summed to count excluded respondents.

In [9]:
n_raw = len(clean)
n_age = int(clean["age_eligible"].sum())
n_final = len(analysis)
sample_flow = pd.DataFrame({
    "stage": ["Original respondents", "Confirmed age 16+", "Complete core responses among eligible respondents"],
    "n_retained": [n_raw, n_age, n_final],
    "n_excluded_at_stage": [0, n_raw - n_age, n_age - n_final],
})
sample_flow["percent_of_original_retained"] = (100 * sample_flow["n_retained"] / n_raw).round(2)
display(sample_flow)

# Supporting descriptive table; no weights have been applied.
country_counts = analysis["country"].value_counts().rename_axis("country").reset_index(name="n")
country_names = pd.DataFrame({"country": raw["country"], "country_name": labels["country"]}).drop_duplicates()
assert country_names["country"].is_unique
country_counts = country_counts.merge(country_names, on="country", how="left", validate="one_to_one")
display(country_counts)

,stage,n_retained,n_excluded_at_stage,percent_of_original_retained
0,Original respondents,25646,0,100.00
1,Confirmed age 16+,25631,15,99.94
2,Complete core responses among eligible respond...,24431,1200,95.26


,country,n,country_name
0,11,1061,Germany
1,4,996,Croatia
2,22,978,Portugal
3,26,978,Spain
4,12,974,Greece
5,9,973,Finland
6,27,972,Sweden
7,25,971,Slovenia
8,15,969,Italy
9,24,966,Slovakia


## 10 Audit contextual availability in the main sample

The main inclusion rule still depends only on age and the five core responses. Do not call `dropna()` on the entire expanded dataframe. For an age comparison, use valid age; for a support comparison, use valid support responses. Report the denominator and missingness for each.

Counts below document sample composition and available data. They do not yet show whether a contextual variable explains or changes the contact-loneliness relationship. That requires the planned EDA or later modelling.

In [10]:
context_analysis_cols = list(category_rules) + ["close_friends_n", "close_family_n"] + list(household_ranges)
context_availability = pd.DataFrame({
    "valid_n": analysis[context_analysis_cols].notna().sum(),
    "missing_n": analysis[context_analysis_cols].isna().sum(),
    "missing_percent": (100 * analysis[context_analysis_cols].isna().mean()).round(2),
})
context_availability.index.name = "variable"
display(context_availability)

composition_tables = []
for col in ["gender_label", "relationship_label", "work_status_label", "education_label",
            "municipality_type_label", "age_group", "close_friends_group"]:
    counts = analysis[col].astype("object").fillna("Missing / non-response").value_counts()
    table = counts.rename_axis("category").reset_index(name="n")
    table.insert(0, "variable", col)
    table["percent_of_main_sample"] = (100 * table["n"] / len(analysis)).round(2)
    composition_tables.append(table)
sample_composition = pd.concat(composition_tables, ignore_index=True)
display(sample_composition)

,valid_n,missing_n,missing_percent
variable,,,
gender,24381,50,0.20
relationship,24324,107,0.44
work_status,24431,0,0.00
education,24335,96,0.39
municipality_type,24271,160,0.65
health_general,24296,135,0.55
health_condition,21399,3032,12.41
family_meet_face,24261,170,0.70
family_meet_tele,24320,111,0.45


,variable,category,n,percent_of_main_sample
0,gender_label,Female,12645,51.76
1,gender_label,Male,11637,47.63
2,gender_label,In another way,99,0.41
3,gender_label,Missing / non-response,50,0.20
4,relationship_label,Married or cohabitating,13160,53.87
5,relationship_label,Single,5202,21.29
6,relationship_label,In a relationship,3503,14.34
7,relationship_label,Separated or divorced,1803,7.38
8,relationship_label,Widowed,656,2.69
9,relationship_label,Missing / non-response,107,0.44


## 11 Validate and export
Save two datasets: a cleaned version retaining all rows and flags, and the complete-case sample for the primary descriptive comparison. Save aggregate audit tables for your presentation. Re-running this cell replaces these generated outputs, never the raw inputs. Survey weights are retained but unused until their definitions have been checked. The separate `weight` variable is not selected as a survey weight.

In [11]:
assert len(clean) == len(raw)
assert analysis["id_pers"].is_unique
assert not analysis[core_cols + ["ucla_total", "age"]].isna().any().any()
assert analysis[contact_cols].isin(range(1, 8)).all().all()
assert analysis[ucla_cols].isin([1, 2, 3]).all().all()
assert analysis["ucla_total"].between(3, 9).all()
assert clean["ucla_total"].isna().equals(clean["ucla_items_answered"].lt(3))
assert analysis["age"].ge(16).all()

clean.to_csv(PROCESSED_DIR / "eu_loneliness_cleaned_with_flags.csv", index=False)
analysis.to_csv(PROCESSED_DIR / "eu_loneliness_analysis_sample.csv", index=False)
missing_audit.to_csv(TABLES_DIR / "missing_response_audit.csv")
age_audit.to_csv(TABLES_DIR / "age_audit.csv", index=False)
sample_flow.to_csv(TABLES_DIR / "sample_flow.csv", index=False)
codebook_observed.to_csv(TABLES_DIR / "observed_code_labels.csv", index=False)
country_counts.to_csv(TABLES_DIR / "analysis_country_counts.csv", index=False)

# Contextual outputs use the main analysis sample unless explicitly labelled.
context_missing_audit.to_csv(TABLES_DIR / "context_missing_audit_raw_sample.csv")
count_audit.to_csv(TABLES_DIR / "close_relationship_count_audit_raw_sample.csv", index=False)
context_availability.to_csv(TABLES_DIR / "context_availability_main_sample.csv")
sample_composition.to_csv(TABLES_DIR / "sample_composition.csv", index=False)

assert analysis["id_pers"].equals(clean.loc[clean["age_eligible"] & clean["core_complete"], "id_pers"])
assert analysis["age_group"].notna().all()
assert analysis["close_friends_group"].isna().equals(analysis["close_friends_n"].isna())

for path in [VALUES_PATH, LABELS_PATH]:
    assert hashlib.sha256(path.read_bytes()).hexdigest() == raw_hashes[path.name]
print("Validation passed. Raw files are unchanged.")
print("Prepared datasets:", PROCESSED_DIR)
print("Audit tables:", TABLES_DIR)

Validation passed. Raw files are unchanged.
Prepared datasets: c:\Users\ladde\Documents\GitHub\PresentationDraft1\data\processed
Audit tables: c:\Users\ladde\Documents\GitHub\PresentationDraft1\results\tables


## 12 Explain the decisions in your own words

Before presenting, make sure every group member can explain:

- Why 999 must be recoded even though the CSV cells are not blank.
- Why valid extreme loneliness scores remain in the data.
- Why the three-item sum requires three valid answers.
- Why the same complete-case sample is used to compare both contact modes.
- Why excluding incomplete cases can introduce bias even with a large retained sample.
- Why age exclusions are a documented project decision that could be examined in a sensitivity analysis.

Next notebook: **02_exploratory_data_analysis.ipynb**. Use the prepared data to describe distributions and plot loneliness across ordered contact categories. Specify whether results are unweighted or use a verified survey weight. These observational patterns do not establish causation.

For later predictive work, preserve a held-out test set before using outcome relationships for feature/model selection. Fit learned preprocessing only on training data. This notebook uses fixed response-code rules and row-wise scoring; it does not learn imputation or scaling parameters, train a model or evaluate predictions.

The exported respondent-level CSVs are local project working files. Check the data reuse terms before redistributing them publicly; aggregate audit tables can support your slides.

### Contextual interpretation
Explain why each retained variable is relevant before looking for strong correlations. Nominal codes have no numerical order. Support and health may overlap with the outcome or lie on a possible pathway, so relevance does not make them automatic confounders. Subgroup differences are descriptive and do not themselves establish a statistical interaction. Keep optional missingness separate from the main inclusion rule.

The existing Notebook 2 still runs on this expanded file and reproduces the original main results. It needs additional sections to analyse the new contextual variables.
